# ⚽ EZStats — Living Product Plan

> **Single source of truth.** All plans, updates, and status changes go here only.

---
## 🎓 Phase 0-T · Model Retraining Plan (June 2026) — 4 Colab notebooks

> **Why past retrains felt useless:** same dataset + same config = identical weights. Each notebook below changes an **input** (more data / bigger model / augmentation / higher resolution), so none is a bit-identical re-run. All train **directly onto Drive** and **resume** after a Colab disconnect.

### The 4 notebooks (start one by one — order = priority)

| # | Model | Notebook | Data source | What changed | ROI |
|---|-------|----------|-------------|--------------|:---:|
| 1 | **Player/GK/Ref detector** | `train_player_detector.ipynb` | **upload `detector.zip`** (your 298-img set) | yolov8**n→m**, 50→**150 ep**, +aug, cosine | **HIGH** |
| 2 | **Event spotter** | `train_pcbas2026_player.ipynb` | SoccerNet PCBAS-2026 (HF download) | player-centric Transformer, 9 classes | **HIGH** |
| 3 | **Pitch keypoints** | `train_pitch_keypoint_detector_v2.ipynb` | Roboflow (local `keypoints/` is **empty**) | imgsz **640→1280**, 100→**200 ep**, cosine | **MED** |
| 4 | **Ball detector** | `train_ball_detector_v2.ipynb` | **upload `ball.zip`** (3956 imgs = v4, 6× old) | 50→**80 ep**, +aug, cosine | **LOW–MED** |

> Originals `train_ball_detector.ipynb` / `train_pitch_keypoint_detector.ipynb` kept as fallback reference. **Tracking** has no notebook — not a trainable model; fixed by #1 + the SigLIP merger (§1-A, code).

### Uploading your own data (Drive)
Claude zips the local datasets to your Desktop. You upload them once:
- `detector.zip` (18.8 MB) → `MyDrive/ezstats/datasets/` → used by #1
- `ball.zip` (208 MB) → `MyDrive/ezstats/datasets/` → used by #4
- #3 pitch pulls from Roboflow (no local keypoint data); #2 downloads from HuggingFace.

To re-zip later (Windows PowerShell):
`Compress-Archive -Path "data\datasets\roboflow\detector\*" -DestinationPath "$env:USERPROFILE\Desktop\detector.zip"`

### Validate-before-swap rule (critical)
Every notebook saves to a **new** path and keeps the old model. After training: test side-by-side on `08fd33_4.mp4`; only switch the pipeline default if it *visibly* wins. Claude does the code switch only after you confirm — **never change `src/` model paths on a hunch.**

---
### 🟢 Beginner Colab runbook (same for every notebook)
1. Drive → folder `ezstats` → subfolder `datasets`. Upload the data zip(s) into `datasets/`.
2. Upload the notebook into `ezstats/` → double-click → opens in **Colab**.
3. **Runtime → Change runtime type → GPU (T4) → Save**.
4. Run cells **top to bottom**, one at a time; read the markdown above each.
5. Model auto-saves to `MyDrive/ezstats/runs/<name>/weights/best.pt`.
6. **Disconnected?** Re-run the train cell — it resumes from the Drive checkpoint.
7. Download `best.pt` → put under `artifacts/...` at a **NEW** path → tell Claude to validate.

**Free keys:** #1/#4 need nothing extra (data is uploaded); #3 needs a Roboflow key; #2 needs a HuggingFace token + SoccerNet NDA password.

**Checklist**
- [ ] #1 Player detector — upload `detector.zip` → train (yolov8m, 150ep) → validate → swap decision  ← **KICK OFF FIRST**
- [ ] #2 Event spotter (PCBAS-2026) → download `model.pt` → then Phase 0-B code rewrite
- [ ] #3 Pitch keypoints (imgsz 1280, Roboflow) → validate → swap decision
- [ ] #4 Ball detector v2 — upload `ball.zip` → only swap if it beats old recall (0.83)


## 📋 Session Rules

| # | Rule |
|---|------|
| 1 | **Read this file (`docs/product_plan.ipynb`) first** every Claude session before doing anything |
| 2 | **Finish one step fully** (test it, verify it) before starting the next |
| 3 | **Stuck on a step?** Diagnose and fix it — never skip ahead |
| 4 | **End of session** — update checkboxes and statuses here |

## Where We Are Right Now - updated 2026-09-10

**Deadline June 22 2026 has passed.** Worker last touched 2026-06-26; backend has continued to 2026-09-01.

### Known Good Run - the demo
`demo/20260608_224414/` - clip `data/raw/08fd33_4.mp4` (30 s, Bundesliga, fixed camera).
Produced by **Pipeline C** (`run_pipeline_C_newplayer_oldball.ps1`) plus the manual post-step
`python -m src.ez_worker.postprocess.report_cleanup <run_dir> --apply`.

| Component | Model |
|---|---|
| Player/GK/Ref | `artifacts/training/player_detector_v2/weights/best.pt` (yolov8m, 150 ep) |
| Ball | `artifacts/ball/football-ball-detection.pt` - **old** one (v2 is worse for events) |
| Pitch keypoints | `artifacts/pitch/football-pitch-detectionV2.pt` |
| Tracker | `configs/bytetrack_football.yaml` |
| Teams | SigLIP + UMAP + KMeans, re-fitted per video |

Result: 39 raw -> **26 players (13/13)**, **8 events**, 0 false events, 0 phantom IDs.
**Verified reproducible 2026-06-18** (`outputs/20260618_001115` is byte-identical except `match_id`).

### Reality check on other videos
Pipeline C was **tuned on one clip and does not transfer**. See the Root Cause Analysis below.

### LOCKED - Do Not Touch
- `stats_video.py` per-frame **TeamClassifier** (get_crops ~L114, fit ~L317-323) - fragile, every "improvement" has broken it
- `report_cleanup.py` protected-actor list - events depend on it
- `configs/bytetrack_football.yaml` - *(UNLOCKED 2026-09-10: see RC-2, it is measurably misconfigured)*

---
## Root Cause Analysis - 2026-09-10

> Why the demo works and nothing else does. All four causes are **configuration / algorithm**
> problems, **not** model-quality problems. No retraining is required to fix any of them.

### Evidence base

| Video | Res / fps | Camera pan (mean / p90) | Cuts | Drift from homography frame (median / p90) | Pitch KPs | Result |
|---|---|---|---|---|---|---|
| **08fd33_4** (demo) | 1080p / 25 | 65 / 149 px/s | 0 | **434 / 937 px** | 17 | OK - 26 players, 8 events |
| **BrightonGoal_clean** | 720p / 25 | 68 / 108 px/s | 0 | **277 / 883 px** | 12 | run 2026-09-10 |
| 19PassesAndMasonGoal | 1080p / 25 | 190 / 372 px/s | 3 | **1505 / 3245 px** | 11 | BAD - 59 players, 5 passes of 19 |
| leo_messi_30pass | 1036p / 60 | 147 / 298 px/s | 2 | **3106 / 3993 px** | 13 | WORST - 59 players, 1 pass of 30 |

*Pan and drift measured by cumulative phase correlation on downscaled frames - absolute values
approximate, relative magnitudes are the finding. Cuts detected by histogram Bhattacharyya distance.*

**Cuts are NOT the problem** (Messi is one continuous shot for 99 of its 108 s). **Camera pan is.**

---

### RC-1 - One static homography is applied to the entire video  <- BIGGEST
`rerun_events.py` builds **a single** homography from `pitch_keypoints.json` -> `best_frame`,
then projects *every* frame's players and ball through it. `pitch_keypoints.json` stores exactly
one frame's keypoints (`best_frame`, `visible_keypoint_count`).

The camera then pans away from that frame. On Messi the camera travels a **median 3106 px** - more
than a full frame width - from where the homography was calibrated. Positions in "pitch cm" are
wrong by tens of metres, yet the log confidently prints `Event detection mode: PITCH (cm)`.

Possession radius, pass speed, shot speed and distance-covered are all computed from those
coordinates. **Every downstream number inherits the error.**

- `stats_video.py` **already computes a per-frame homography** for the radar overlay - it is simply
  never passed to the event detector.
- `pipeline.py:64` calls `detect_events()` with **no pitch coordinates at all**, so `analyze` always
  runs pixel-fallback; only the later `rerun_events.py` pass uses cm.
- This is plan item **1-D** ("Plumb pitch coordinates into detect_events()"), still unchecked.

### RC-2 - ByteTrack's second association stage is dead
Verified against the installed `ultralytics/trackers/byte_tracker.py`:

1. `--detection-confidence 0.20` (YOLO) **equals** `track_high_thresh: 0.20`. YOLO never emits a box
   below 0.20, so the low-score bucket `[track_low_thresh 0.05, track_high_thresh 0.20)` is
   **always empty**. ByteTrack's second association - the entire reason it beats SORT, recovering
   occluded players from weak detections - is fed nothing.
2. `fuse_score: true` makes the cost `1 - IoU x score`, and stage 2 uses a hardcoded `thresh=0.5`.
   A detection with score < 0.20 would need `IoU > 2.5` to match - **mathematically impossible**.
3. `new_track_thresh: 0.20` means any unmatched detection immediately mints a fresh ID.

Measured consequence - Brighton: **1220 track IDs minted for ~12 concurrent players** over 908
frames; median surviving track lifespan **86 frames vs 436 on the demo**; 20% of tracks live < 1 s.
The postprocess then hides this by discarding short tracks, which also discards real players.

All three configs (`bytetrack_football`, `bytetrack_football_v2`, `botsort_football`) share
`new_track_thresh: 0.20`. Past experiments only varied `match_thresh` and ReID - **this lever was
never tested.** Note also that the `bytetrack_football_v2.yaml` comment is backwards: `match_thresh`
0.85 -> 0.75 makes matching *stricter*, not "more tolerant of partial occlusion".

### RC-3 - Event thresholds partly bound to scale  [CORRECTED 2026-09-10]

> **This entry was initially overstated and is corrected here.** The first version
> claimed `_STANDARD_PLAYER_HEIGHT_PX = 70.0` inflates all distances on non-1080p
> video. That is **wrong**: `_closest_player` divides by **each player's own bbox
> height** (`raw * (70 / p_h)`), so 70 is only a unit constant and it cancels
> against the threshold - the distance metric is already scale-free. Measured
> median player heights: demo 59.5 px, Brighton 65.2 px (720p but zoomed in, so
> players are *bigger* than the 1080p demo, not smaller), Mason 92.6, Messi 109.7.

What is genuinely wrong, and much narrower:

- **Ball speed is not scale-normalised.** `_compute_ball_velocity` returns raw
  px/frame with no player-height ruler, while distances and travel *are*
  normalised. A Messi player is 1.84x taller than a demo player, so the same real
  ball speed yields 1.84x the px/frame and the pass/shot gates mean something
  different on every clip.
- `_VEL_SMOOTH_WINDOW = 5` is fixed in frames, not fps-scaled: 200 ms at 25 fps,
  83 ms at 60 fps.
- `_MAX_BALL_SPEED_PX_FRAME = 100.0` is a raw-pixel cap with the same issue.

**Priority note:** these only bite in **pixel-fallback** mode, and every run so far
(demo, Brighton, Mason, Messi) ran in **PITCH (cm)** mode - confirmed by the
presence of `events_pixel_fallback.json`, which `rerun_events.py` only writes when
pitch mode succeeded. So RC-3 is a **latent** bug, not the active cause of bad
events. **RC-1 is the dominant event problem.** Fix RC-3 for robustness, not first.

### RC-4 - Pitch keypoint detection is sparse and single-frame
Brighton logged `0 keypoints` on most sampled frames; only 12 keypoints at the best frame (demo: 17).
Six pairs are the minimum for the homography, so the margin is thin. SOTA uses **multi-frame**
keypoint aggregation.

---
## Phase 1-R - Improvement Roadmap (NO retraining)

> **Ground rule (user, 2026-09-10):** training is off the table for now - validation metrics were
> already good (player mAP50 0.99) while real-world behaviour was wrong, which is the signature of a
> *deployment/config* problem, not a model problem. RC-1..RC-4 confirm this. Training is
> reconsidered only if these are exhausted.

### Tier 1 - config & algorithm (highest value, hours not days)

| # | Fix | Targets | Risk |
|---|---|---|---|
| R1 | **Tracker config**: `detection-confidence` 0.20->0.05, `track_high_thresh` 0.20->0.50, `new_track_thresh` 0.20->0.60, `fuse_score` true->**false** | RC-2 | Low - YAML only, instantly revertible |
| R2 | **Per-frame homography into events**: reuse `stats_video.py`'s per-frame H; store `pitch_keypoints_per_frame.json`; fall back to nearest valid frame | RC-1 | Medium - new data path, guard with try/except |
| R3 | **Scale-normalised thresholds**: derive the player-height constant *per video* from median bbox height; normalise ball speed by it; scale smoothing windows by fps | RC-3 | Medium - changes event numbers, must re-verify demo's 8 events |

**Verification gate for every change: `08fd33_4` must still produce the same 8 events.**
That is the regression test. `outputs/20260618_001115` is the byte-identical reference.

### Tier 2 - classical image processing (from the DIP class, Ch2/4/5/6/7)

| # | Technique (source) | Applied to | Why it should work |
|---|---|---|---|
| R4 | **HSV grass mask + morphological opening/closing** (Ch6 p49; Ch7 p14/p18 - *opening eliminates small islands, closing fills gaps*) | Reject crowd / bench / advertising detections **before** tracking; constrain pitch keypoints | `report_cleanup` already removes off-pitch tracks *post-hoc* via homography - doing it in pixel space needs no homography, so it works on the videos where RC-1 breaks |
| R5 | **Team classification by hue histogram** (Ch6 p49 - *"segmentation can be performed on the H component"*) | Replace/augment SigLIP team clustering | SigLIP is the known-broken component; the crude torso-HSV vote in `report_cleanup` **already beats it**. Hue is invariant to the shadow/brightness changes that break RGB and deep embeddings. Grass mask (R4) fixes the "green bleed" that made the earlier colour test too noisy |
| R6 | **Top-hat transform** (Ch7 p50 - *extracts bright spots/ridges*) | Ball candidate verification when YOLO misses | The ball is a small bright blob on uniform green - the textbook top-hat case. Use as gap-filler, **not** a replacement (YOLO already hits 93% on Brighton) |
| R7 | **Median / adaptive-median filter on the ball trajectory** (Ch4 p15; Ch5 adaptive median) | Kill impulse outliers in ball path | Already partly present (5-frame rolling median on velocity); extend to position and make the window fps-scaled |
| R8 | **Phase-correlation camera-motion compensation** (Ch3 geometric transforms) | Feed global motion to the tracker / warp the homography per frame | Already prototyped 2026-09-10 - it produced the pan table above. BoT-SORT's `gmc_method: sparseOptFlow` does this natively and is already in `botsort_football.yaml`, unused |

> **Anti-recommendation:** do **not** pre-filter/smooth frames before YOLO. The detector was
> trained on unfiltered broadcast frames; blurring removes exactly the high-frequency detail it keys
> on. Classical filtering belongs on **crops** (team colour), **masks** (pitch), and **trajectories**
> (ball) - never on the detector's input.

### Tier 3 - architecture (aligned with published SOTA)

| # | Direction | Source |
|---|---|---|
| R9 | **Player-centric ball-action spotting** - attribute actions to players via player-ball graph reasoning rather than a hand-tuned state machine | FOOTPASS, arXiv 2606.09679; Entity-Aware Sequence Transduction, arXiv 2608.01696 |
| R10 | **Ball-free possession-path inference** - infer possession from *player trajectories only* via CRF + Viterbi, removing the ball-detection dependency | PathCRF, arXiv 2602.12080 |
| R11 | **Deep-EIoU + OSNet ReID tracking** + multi-frame keypoint homography | SoccerNet GSR 2025 winner, arXiv 2504.06357 |

### Strategic finding - what our input should be
**Veo films the entire pitch at all times from a fixed camera**; it does not analyse panning
broadcast footage. Their tracking works *because* the camera does not move. Our own measurements say
the same thing: quality collapses above ~150 px/s p90 pan.

-> **Target fixed wide-angle club-camera footage** (exactly the SE Asia club market in Phase 6).
Broadcast highlight edits are the hard case that even commercial vendors avoid. This should be
stated as a scope decision, not hidden as a limitation.

---
## Brighton control experiment - 2026-09-10 (run `outputs/20260910_182918`)

**Verdict: NOT demo quality. Do not add to `demo/`.** But it is the most useful run we have done,
because it *isolates* the failure modes: camera pan and homography were FINE here, and the output
was still bad. That rules out RC-1 as the sole cause and exposes two new ones.

Setup: `BrightonGoal.mp4` trimmed to its continuous 36.3 s segment -> `BrightonGoal_clean.mp4`,
Pipeline C, then `report_cleanup --apply`. Total runtime **37:53** (analyze 5:41, cluster 1:34,
**render-stats-video 29:57**, rerun-events 0:02).

| Metric | Result | Verdict |
|---|---|---|
| Ball detected | 846/908 frames (93%) | GOOD |
| Homography | valid, reproj err 4.2 cm | GOOD - but only **5 inliers of 12**, the bare minimum the gate allows |
| Camera drift from best frame | 277 px median | GOOD - comparable to the demo |
| Raw track IDs minted | **1220** for ~12 concurrent players | BAD (RC-2) |
| Players after cleanup | 57 -> 34 -> **25** | comparable to demo's 26 |
| Events | 8 (4 pass, 2 touch, 1 header, 1 long_ball) | plausible chain, but... |
| Teams | **19 / 6**, every event actor `T1#` | BROKEN (RC-5) |
| Possession | **100 / 0** | BROKEN |
| The goal | **not detected** - no shot, no goal | see note below |

### RC-5 - The referee filter deletes any team wearing a dark kit  <- NEW
Brighton (blue/white stripes) played **Fulham in a black away kit**. `report_cleanup`'s referee
heuristic removes tracks with a *dark, low-saturation torso*. It removed 9 tracks. Visual inspection
of those crops confirms they are **nine Fulham outfield players**, one with a visible squad number -
not referees. **One entire team was deleted as officials.**

Same root shape as the rest: `correct_teams_by_color` hard-codes **GREEN (H40-78, S>55) vs WHITE
(V>180, S<55)** from `08fd33_4`. Brighton/Fulham match neither, so the colour correction silently
no-ops while the referee filter over-fires.

### RC-6 - SigLIP team clustering fails on this footage
Even *before* cleanup the raw report was **49 / 8**. The clip is heavily zoomed (only 2-3 players
visible for long stretches), so the per-video KMeans fit has very little to separate. This is the
already-known-fragile component (see LOCKED list) failing on new input, independent of RC-5.

### Note - the goal was trimmed off (operator error, 2026-09-10)
The scene-cut detector found a cut at frame 909 and the clip was trimmed there, on the assumption
that the cut marked the switch to replay. Frame inspection at t=40 s shows the opposite: the cut is
the switch to the **wide angle in which the goal is scored**. The trimmed segment contains only the
build-up. If a goal example is wanted, re-run on the **full** `BrightonGoal.mp4`, accepting the cut,
and use `report_cleanup --goal-frame N` to mark it.

### What this experiment proves
1. Low camera pan is **necessary but not sufficient** - RC-1 is real but not the only blocker.
2. RC-2 (tracking fragmentation) is **independent of camera pan** - 1220 IDs with a good homography.
3. The colour heuristics in `report_cleanup` are **overfitted to one match's kit colours** and can
   silently delete a team. Any per-video colour logic must be **learned from that video**, never
   hard-coded (this is exactly what R5's hue-histogram approach fixes).
4. `render-stats-video` is **79% of runtime** (29:57 of 37:53) - the correct target for the Colab
   GPU migration.

---
## Pipeline v2 - build log, 2026-09-10

New files: `run_pipeline_v2.ps1` (local), `run_pipeline_v2.py` (portable, used by
Colab so both platforms run ONE definition), `configs/bytetrack_football_v3.yaml`,
`docs/run_on_colab.ipynb`. `report_cleanup` is now a pipeline STEP, so every run
emits `match_report_merged.json` without a manual command.

### Fixes shipped

| Fix | Change | Evidence |
|---|---|---|
| **RC-5** referee cap | `MAX_REFEREES = 4`; above that the dark-torso group is a KIT, remove nobody | Demo still removes its 3 real refs; Brighton keeps all 9 Fulham players |
| **RC-6** learned kits | `learn_kit_buckets()` - k-means on a saturation-weighted hue vector, **k=3** so the keeper cannot capture a seed. Runs ONLY as a fallback when the hard-coded GREEN/WHITE path fails | Verified by crop montage: cluster A = Fulham black (squad numbers visible), B = Brighton stripes |
| **possession** | `recompute_possession()` - share of FRAMES nearest the ball, radius in player-heights (scale-free) | Demo 50.0/50.0 (which was just 1 touch each) -> 59.3/40.7; Brighton 100/0 -> 18.1/81.9 |
| **RC-1** per-frame homography | `--per-frame-stride` writes `pitch_keypoints_per_frame.json`; `rerun_events.py` uses the NEAREST frame's matrix. Best-frame output kept byte-identical | Brighton: **73 valid matrices** vs 1. Demo with no per-frame file: **8 events, identical to the committed reference** |
| **RC-2** tracker | `bytetrack_football_v3.yaml`: `fuse_score false`, `new_track_thresh 0.60`, `track_high_thresh 0.50` | Max raw IDs minted: demo 510 -> 142, Brighton 1220 -> 263 |

### NEGATIVE RESULT and the correction it forced

The first v2 paired the tracker config with `--detection-confidence 0.10`, on the
theory that the low-score bucket needed weaker detections to exist at all.

It cut ID churn as predicted **but made the reports worse**:

| | Demo v1 | Demo v2 (conf 0.10) | Brighton v1 | Brighton v2 (conf 0.10) |
|---|---|---|---|---|
| Max IDs minted | 510 | **142** | 1220 | **263** |
| Players after cleanup | 26 | **31** | 34 | **40** |
| Events | 8 | 9, incl. a **double clearance at f410 and f411** | 8 | 8, more coherent |

Crop inspection of the extra Brighton tracks: a mix of genuinely-new real players
(better recall) AND crowd/motion-blur junk (#11, #53 are spectators). Because
tracking was now *stable*, that junk survived as long-lived tracks instead of
being churned away - the fix made the noise more persistent.

**Diagnosis of my own error:** the second association does not need weaker
detections. It needs the **TRACKER's** high threshold to sit **above the
DETECTOR's** threshold. Keeping YOLO at 0.20 with `track_low_thresh: 0.20` and
`track_high_thresh: 0.50` gives a bucket of [0.20, 0.50) - real-but-uncertain
boxes, exactly ByteTrack's design - without admitting 0.10-confidence blur.
v3.yaml and both runners were corrected to detector conf **0.20**.

### Lesson that generalises
Every failure found today has the same shape: **a constant fitted to `08fd33_4`**
- its kit colours (RC-5, RC-6), its possession being 1 touch per side, its single
camera pose (RC-1). The system is not yet a football analyser; it is a very good
`08fd33_4` analyser. Fixing that means **deriving constants from each video**
rather than hard-coding them, which is what `learn_kit_buckets`, per-frame
homography and the player-height radius all now do.

### Open, next in priority order
1. **HSV grass mask + morphological opening/closing (R4)** - reject crowd/bench in
   PIXEL space before tracking. Needs no homography, so it works on exactly the
   clips where RC-1 breaks. This is the fix for the junk-track regression above.
2. Re-verify the demo gate after the conf-0.20 correction.
3. **Messi** (`leo_messi_30pass.mp4`) is the priority test clip - clear resolution,
   pink kit (high saturation, ideal for the hue clustering), contains a real goal,
   AND has the worst measured camera drift (3106 px), so it is the strongest test
   of the RC-1 fix.
4. `BrightonGoal.mp4` **contains no goal** - verified frame by frame, scoreboard
   reads 0-0 (clock 11:30 -> 11:35) through the final frame. Do not use it for a
   goal demo. It is also the noisiest clip (heavy zoom, crowd bleed, motion blur).
5. RC-3 (ball speed not scale-normalised) - latent only; pitch mode masks it.

## 📊 Current Status vs Footovision

| Feature | Footovision | EZStats | Status |
|:--------|:-----------:|:-------:|:------:|
| Player + ball + referee detection | ✅ | works but misses frames, new videos | ⚠️ |
| Homography / top-down pitch | ✅ | not 100% — bad keypoints warp pitch | ⚠️ |
| **Player ID tracking — occlusion** | ✅ stable | **IDs swap when players overlap** | ❌ |
| Team classification | ✅ | breaks on ID re-assign, some frames wrong | ⚠️ |
| Voronoi territorial control | ✅ | wrong — depends on team classification | ⚠️ |
| Basic possession stats | ✅ | wrong — depends on tracking bugs above | ⚠️ |
| Three-phase event detection | ✅ | rewritten, passes wrong due to tracking | ⚠️ |
| Clearance / keeper long ball detection | ✅ | misclassified as pass | ❌ |
| 9-class Transformer (PASS/SHOT/CROSS…) | ✅ | training in Colab | 🔄 |
| Player heatmaps | ✅ | not built | ❌ |
| Per-player speed & distance (meters) | ✅ | pixel hack only — wrong numbers | ❌ |
| Event highlight clips | ✅ | not built | ❌ |
| Pass network | ✅ | not built | ❌ |
| Debug viewer / match_report.json | — | built ✅ | ✅ |
| Cloud API | ✅ | offline only | ❌ |
| Formation detection | ✅ | not built | ❌ |
| xG model | ✅ | not built | ❌ |
| Real-time processing | ✅ | ~20 min / 30s on CPU | ❌ |
| Mobile dashboard | ✅ | not built | ❌ |
| **Affordable SE Asia pricing** | ❌ enterprise | **our core edge** | ✅ |

---
## 🚀 Phase 0 — Fix Before Any Run

### 0-A · Test Rule-Based Events

- [x] Remove broken `--event-model-name` from `run_full_pipeline.ps1` + `run_new_video.ps1`
- [x] Run `.\.run_full_pipeline.ps1` → opened viewer, loaded `match_report.json`
- [ ] **Passes still wrong** → blocked by tracking ID swap bug (fix in Phase 1-A first)
- [ ] Re-run after tracking fix → manually count passes → target within 25% of truth

### 0-B · Rewrite event_spotter.py for PCBAS-2026 ⏸ WAITING FOR TRAINING

> Training notebook: `docs/train_pcbas2026_player.ipynb`  
> Download when done: Colab Drive → `artifacts/training/event_spotter_pcbas2026/model.pt`

**What's wrong with the current code vs what it should be:**

| | ❌ Current | ✅ Should Be |
|---|---|---|
| Architecture | Bi-LSTM | Transformer Encoder (4-layer, 8-head) |
| Classes | 14 BAS-2025 | **9 PCBAS-2026** (different order!) |
| Window | 15 frames | **25 frames** |
| Inference | Global frame features | **Per-player crop features** |
| Output head | Single (action) | **Dual (action + team)** |
| Save path | `event_spotter_bas2025` | **`event_spotter_pcbas2026`** |

**Correct class list (order matters for loading weights):**
```python
BALL_ACTION_CLASSES = [
    'background',  # 0
    'DRIVE',       # 1
    'PASS',        # 2
    'CROSS',       # 3
    'SHOT',        # 4
    'HEADER',      # 5
    'THROW IN',    # 6
    'TACKLE',      # 7
    'BLOCK',       # 8
]
```

**Correct architecture:**
```python
self.proj        = nn.Linear(512, 512)
self.pe          = PositionalEncoding(512, dropout=0.1)
self.encoder     = nn.TransformerEncoder(
    nn.TransformerEncoderLayer(512, nhead=8, dim_feedforward=2048,
                               dropout=0.1, batch_first=True, norm_first=True), 4
)
self.shared      = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.1))
self.action_head = nn.Sequential(nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 9))
self.team_head   = nn.Linear(256, 3)   # 0=background, 1=left-team, 2=right-team
```

**Checklist (do after model.pt is downloaded):**
- [ ] Rewrite `_EventSpotterModel` to Transformer above
- [ ] Update `BALL_ACTION_CLASSES` → 9 PCBAS-2026 classes
- [ ] Update `WINDOW = 25`
- [ ] Update `_CLASS_TO_EVENT` + `_CLASS_THRESHOLDS` for 9 new classes
- [ ] Rewrite inference to be player-centric (per-player crop features, not frame features)
- [ ] Update `fuse_events()` class names
- [ ] Uncomment `--event-model-name` in `run_full_pipeline.ps1` + `run_new_video.ps1`

### 0-C · Debug Dashboard

- [x] `outputs/<timestamp>/match_report.json` written after every pipeline run
- [x] `outputs/viewer.html` — open in browser, load JSON, see all stats visually
- [ ] Re-run after tracking fix → open viewer → confirm passes match video

---
## 🔧 Phase 1 — Make Existing Features Actually Work
*Target: Week 1–2*

### 🔁 1-A · Tracking Stability — SigLIP Tracklet Merger  ← NEXT

> **Decision:** Use BoT-SORT with `with_reid: false` as base tracker.  
> GMC handles camera pan. SigLIP merger handles post-hoc ID fixing.

**Pipeline order after merge-tracklets is wired in:**
```
analyze
  → prepare-appearance        (generates SigLIP crop embeddings)
  → cluster-teams
  → apply-team-clusters
  → merge-tracklets           ← NEW: remap split IDs using SigLIP similarity
  → render-stats-video
  → render-spatial-video
```

**How merge-tracklets works:**
- Reads `tracks_with_teams.json` + SigLIP embeddings from `player_crops/`
- For each pair of tracklets: (a) no time overlap, (b) same team, (c) cosine similarity > 0.82 → remap higher ID → lower ID
- Writes `tracks_merged.json` + rewrites `tracks_with_teams.json`

**Current state:**
- `src/ez_worker/postprocess/siglip_merge.py` — written, numpy `or` bug fixed, NOT wired in
- `cli.py` — reverted (no merge-tracklets command)
- Both run scripts — no merge-tracklets step

**Checklist:**
- [x] ByteTrack v2 tested → ❌ still swaps
- [x] BoT-SORT with ReID tested → ❌ OSNet can't distinguish football players
- [x] Decided: BoT-SORT (ReID off, GMC on) — configs + run scripts updated
- [x] `siglip_merge.py` written + numpy bug fixed (use explicit `is None` checks)
- [ ] Add `merge-tracklets` command back to `cli.py`
- [ ] `render-stats-video` reads `tracks_merged.json` if present (fallback to `tracks_with_teams.json`)
- [ ] Add step to `run_new_video.ps1` + `run_full_pipeline.ps1` after `apply-team-clusters`
- [ ] Run on `outputs/20260508_211913` run dir → open `stats_video.mp4` → confirm IDs stable
- [ ] Open viewer → confirm max ID ≤ 30, passes make sense

### 🎯 1-B · Event Detection — Fix False Passes + High Ball  ✅ DONE 2026-05-12

**Changes made:**
- `possession_distance_threshold_px` 65 → 50 (tighter possession zone)
- `pass_min_speed_px_per_s` 150 → 200 (higher speed needed to enter IN_FLIGHT)
- `ball_direction_change_min_deg` 20 → 35 (stronger direction change for slow-possession pass)
- **Arc detection** (`_is_high_arc`): fits parabolic trajectory to ball y-positions during IN_FLIGHT
  - Peak (min y in image) must occur in middle 15–85% of flight = genuine arc
  - Arc height > 4% of frame height = real high ball
- **`clearance` event**: flight > 1.2s AND arc detected AND no receiver → replaces `shot_attempt` for GK kicks
- **`long_ball` event**: flight > 1.2s AND arc detected AND receiver found → replaces `pass` for high balls
- **Manual track merge**: place `manual_track_merges.json` in run dir, e.g. `[[209, 10]]`
  - Remaps track 209 → 10 in tracks, stats, and events.json
  - Combines numeric stats (touches, passes, shots, distance) for merged tracks
- **match_report.json** now rebuilt after `apply-team-clusters` → fixes team showing as "?"
  - Includes new `total_long_balls` and `total_clearances` in summary

**Checklist:**
- [x] Tighten possession distance (65 → 50px)
- [x] Raise pass speed threshold (150 → 200 px/s)
- [x] Raise direction change threshold (20 → 35°)
- [x] `clearance` event type implemented
- [x] `long_ball` event type implemented
- [x] Physics arc detection using ball y-trajectory
- [x] Manual track merge via `manual_track_merges.json`
- [x] match_report.json rebuilt after apply-team-clusters (fixes "?" team)
- [ ] Run on `19PassesAndMasonGoal.mp4` → manually count passes → target within 25% of truth
- [ ] Verify clearance fires for keeper long ball, not shot_attempt
- [ ] Verify no false passes from loose ball rolling near multiple players
- [ ] Add `clearance`/`long_ball` badge styles to `viewer.html`

### 🧠 1-B2 · GCN-Inspired Event Detection (Spatial Graph) ← IN PROGRESS

> Reference:   
> Paper: Rana 2021 — Event Detection in Football using Graph Convolutional Networks (arXiv:2301.10052)

**Rule-based spatial graph adaptations (no training needed yet):**

| Fix | Problem Solved |
|-----|----------------|
| Self-reception prevention | False A→A pass when ball bounces back to owner |
|  | False pass from instant dribble/toe-poke |
|  graph helper | Multi-player spatial context around ball |
| Arc detection + clearance/long_ball | GK kick misclassified as shot |

**Checklist:**
- [ ] Implement self-reception prevention in IN_FLIGHT phase
- [ ] Add  param to  + config + pipeline
- [ ] Add  helper function
- [ ] Run on  → verify < 20 passes for 30s clip, no A→A events
- [ ] Run on  → manually count 19 passes → events.json within 25%
- [ ] Create  reference doc

**If rule-based not accurate enough → train GCN:**
- Annotate 500+ events across 3+ videos using viewer
- 2-layer GCN(64,64) + NetVLAD++(T=10s) + multi-label classifier
- Input: per-node (x, y, team_one_hot) at 25fps

### 🔍 1-C · Detection Quality Gate
> Current detector is YOLOv8n trained on only 298 images of Bundesliga footage.  
> Everything else depends on detection being reliable.

- [ ] Players detected consistently — no player vanishing for >10 consecutive frames
- [ ] Ball detected in ≥70% of frames
- [ ] All 22 players on pitch detected at same time

> **If gate fails → retrain detector first:**
> - Annotate frames from failing video in Roboflow format
> - Upgrade `yolov8n` → `yolov8s` for better accuracy
> - Target: 1000+ images, 4 classes (player / goalkeeper / referee / ball)
> - `yolo detect train model=yolov8s.pt data=data/datasets/roboflow/detector/data.yaml epochs=100`

### 🗺️ 1-D · Homography — Make It Stable

- [ ] Fix H averaging in `stats_video.py` — compute `np.mean(list(H_deque), axis=0)` (currently just uses last H)
- [ ] Add H quality check — `abs(np.linalg.det(H)) > 0.1` before accepting
- [ ] Clip pitch projections to `[0, 10500] × [0, 6800]` cm after `perspectiveTransform`
- [ ] Plumb pitch coordinates into `detect_events()` via `pipeline.py` (unlocks cm-space event detection)
- [ ] Visual check: Voronoi on `08fd33_4.mp4` looks geometrically correct

### 👕 1-E · Team Classification — Reduce Misassignment

- [ ] Fix GK exclusion — filter by pitch position (closest to goal post) not just detection label
- [ ] Add UMAP guard — if `n_samples < 20`, fall back to cosine similarity clustering
- [ ] Add temporal team lock — once a `track_id` is assigned a team, never flip it mid-track
- [ ] Visual check: team colors stable for ≥90% of frames on both test videos

### 📈 1-F · Possession Stats — Real Numbers

- [ ] Replace pixel distance in `stats.py` with pitch-coordinate distance in meters
- [ ] Validate: possession % on `19PassesAndMasonGoal.mp4` should not be 50-50

---
## ✨ Phase 2 — First Sellable Features
*Target: Week 3–4*

### 🌡️ Player Heatmaps
- [ ] Accumulate `(x_cm, y_cm)` per `tracker_id` in render loop
- [ ] New: `src/ez_worker/analytics/heatmap.py` — Gaussian KDE on 105×68 pitch grid
- [ ] Output PNG per player + team aggregate
- [ ] Add heatmap paths to `match_report.json` + show in `viewer.html`

### 🏃 Per-Player Speed & Distance
- [ ] Distance = sum of Euclidean distance in cm between frames → meters
- [ ] Max speed = `max(Δcm / Δseconds)`, capped at 1200 cm/s (43 km/h)
- [ ] Add to `TrackStats` and `match_report.json`

### 🎬 Event Highlight Clips
- [ ] New: `src/ez_worker/io/clip_exporter.py`
- [ ] On SHOT / GOAL / CORNER / FREE KICK — cut `[event_frame - 2s : event_frame + 3s]`
- [ ] Named: `GOAL_45m23s.mp4`, `SHOT_32m11s.mp4`
- [ ] Add clip paths to events in `match_report.json`

### 🕸️ Pass Network
- [ ] New: `src/ez_worker/analytics/pass_network.py`
- [ ] Directed graph: `player_A → player_B` count from PASS events
- [ ] Output JSON edge list + static pitch diagram image
- [ ] Show in `viewer.html`

### ✏️ 2-A · Manual Player ID Remapping + Roster Tagging

> **Problem:** ~2 players per run get wrong IDs after overlap. User needs to fix before final stats.  
> **Solution:** Corrections panel in viewer.html → download  → re-run 

**Format of :**


**Checklist:**
- [ ] Add corrections panel to  (team override dropdown, merge input, name input)
- [ ] Save Corrections button → downloads  (pure JS, no server)
- [ ]  reads  at start → applies overrides + names
- [ ] Player names appear in  players array + viewer table
- [ ] Test: correct 2 known wrong ID switches → re-run apply-team-clusters → verify stats


---
## 📱 Phase 3 — Friend's Frontend Ready
*Target: Week 5–6*

- [ ] Hand `match_report.json` schema to frontend friend
- [ ] Test on `BrightonGoal.mp4` — first untested video — no crashes, goal detected
- [ ] Heatmaps + clips + pass network all working end-to-end

---
## ☁️ Phase 4 — Cloud API
*Target: Week 7–8*

- [ ] New: `src/ez_worker/api/main.py` — FastAPI
- [ ] `POST /analyze` → upload video → `job_id`
- [ ] `GET /job/{id}` → status + download links
- [ ] Deploy on RunPod / Lambda Labs GPU — 90-min match in < 30 min
- [ ] Simple web UI: drag-and-drop → progress bar → download results

---
## 📐 Phase 5 — Analytics Gap (Month 2)

- [ ] **xG model** — shot `(x_cm, y_cm)` + event class → logistic regression on SoccerNet data
- [ ] **Formation detection** — cluster positions at kickoff/set pieces → 4-4-2 / 4-3-3 etc.
- [ ] **Pressing intensity** — count players within 1000 cm of ball carrier per frame

---
## 🌏 Phase 6 — SE Asia Market Lock-In (Month 3+)

- [ ] Collect 10+ hours Myanmar National League / Thai League footage from clubs
- [ ] Fine-tune YOLO detector on local footage (different jerseys, pitch, lighting)
- [ ] Mobile-responsive dashboard (React Native or PWA)
- [ ] Pricing: **$15 / match** or **$80 / month** per club
- [ ] Onboard first 5 pilot clubs — free trial in exchange for video footage
- [ ] Localization: Burmese + Thai UI

---
## 🗂️ Key Files Reference

| File | Status | Pending Work |
|:-----|:------:|:-------------|
| `configs/botsort_football.yaml` | ✅ | Chosen base tracker — ReID off, GMC on |
| `configs/bytetrack_football_v2.yaml` | ✅ | Tested, rejected — kept for reference |
| `src/ez_worker/postprocess/tracks.py` | 🔧 | SigLIP tracklet merger (Phase 1-A) |
| `src/ez_worker/cli.py` | 🔧 | Add `merge-tracklets` command (Phase 1-A) |
| `src/ez_worker/io/stats_video.py` | 📋 | Read `tracks_merged.json`; accumulate pitch coords; fix H averaging |
| `src/ez_worker/analytics/events.py` | 📋 | Add `clearance` + `long_ball` event types (Phase 1-B) |
| `src/ez_worker/analytics/event_spotter.py` | ⏸ | Full rewrite → PCBAS-2026 Transformer (Phase 0-B, after training) |
| `src/ez_worker/analytics/stats.py` | 📋 | Distance in real meters (Phase 1-F) |
| `src/ez_worker/spatial/view_transformer.py` | 📋 | H quality check + projection bounds clamp (Phase 1-D) |
| `src/ez_worker/appearance/team_assignment.py` | 📋 | GK exclusion fix; UMAP guard (Phase 1-E) |
| `src/ez_worker/outputs/writer.py` | ✅ | match_report.json done |
| `run_full_pipeline.ps1` + `run_new_video.ps1` | ✅ | BoT-SORT active; add `merge-tracklets` step after impl |
| `outputs/viewer.html` | 📋 | Add `clearance`/`long_ball` badge styles (Phase 1-B) |
| `src/ez_worker/analytics/heatmap.py` | 📋 | New — Phase 2 |
| `src/ez_worker/io/clip_exporter.py` | 📋 | New — Phase 2 |
| `src/ez_worker/analytics/pass_network.py` | 📋 | New — Phase 2 |
| `src/ez_worker/api/main.py` | 📋 | New — Phase 4 |